In [1]:
import pandas as pd
import numpy as np


In [2]:
df=pd.read_csv("powerplant_data.csv")
df


,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43
...,...,...,...,...,...
9563,15.12,48.92,1011.80,72.93,462.59
9564,33.41,77.95,1010.30,59.72,432.90
9565,15.99,43.34,1014.20,78.66,465.96
9566,17.65,59.87,1018.58,94.65,450.93


In [3]:
X=df.drop("PE",axis=1)
y=df["PE"]

In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.2,random_state=42)

In [6]:
from sklearn.preprocessing import StandardScaler
ss=StandardScaler()
X_train_scaled=ss.fit_transform(X_train)
X_test_scaled=ss.transform(X_test)

c:\Users\suyas\.conda\envs\ml_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [8]:
import torch    
import torch.nn as nn
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)


In [9]:
from torch.utils.data   import TensorDataset ,DataLoader
train_ds=TensorDataset(X_train_tensor,y_train_tensor)
test_ds=TensorDataset(X_test_tensor,y_test_tensor)


In [10]:
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True)
test_loader=DataLoader(test_ds,batch_size=32)


In [27]:
    class ANN(nn.Module):
        def __init__(self):
            super(ANN,self).__init__()

            self.model=nn.Sequential(
                #1st hidden layer
                nn.Linear(X_train.shape[1],6),
                nn.ReLU(),

                #2nd hidden layer
                nn.Linear(6,6),
                nn.ReLU(),

                #3rd hidden layer
                nn.Linear(6,1)
            )

        def forward(self,x):
            return self.model(x)
    

In [28]:
import torch.optim as optim
model=ANN()
crieterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [30]:
train_losses=[]
val_losses=[]
epochs=100
best_val_loss=float("inf")
for epoch in range(epochs):
    model.train()
    running_loss=0.0

    for xb,yb in train_loader:
        #xb => features of 1 batch
        #yb =>labels of 1 batch
        optimizer.zero_grad() # it is used so that gradient does not get accumalated
        outputs=model(xb) # it is used to predict the output of this batch
        loss=crieterion(outputs,yb) # calculate the loss 
        loss.backward()# back propogation 👉 Computes gradients (how to update weights)
        optimizer.step() #👉 Adjusts weights to reduce loss

        running_loss+=loss.item()
        
    epoch_train_loss=running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)



#validation
    model.eval()
    running__val_loss=0
    with torch.no_grad():
        for xb,yb in test_loader:
            outputs=model(xb)
            loss=crieterion(outputs,yb)
            running__val_loss+=loss
    epoch_val_loss=running__val_loss/len(test_loader)
    val_losses.append(epoch_val_loss)

    if epoch_val_loss<best_val_loss:
        best_val_loss=epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt")
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")


Epoch 1/100 | Train Loss: 58.3618 | Val Loss: 19798962.0000
Epoch 2/100 | Train Loss: 55.0493 | Val Loss: 19573896.0000
Epoch 3/100 | Train Loss: 52.3435 | Val Loss: 19344648.0000
Epoch 4/100 | Train Loss: 49.6055 | Val Loss: 19124074.0000
Epoch 5/100 | Train Loss: 47.2693 | Val Loss: 18884716.0000
Epoch 6/100 | Train Loss: 45.2063 | Val Loss: 18753486.0000
Epoch 7/100 | Train Loss: 43.0407 | Val Loss: 18535390.0000
Epoch 8/100 | Train Loss: 41.0837 | Val Loss: 18384602.0000
Epoch 9/100 | Train Loss: 39.3867 | Val Loss: 18193144.0000
Epoch 10/100 | Train Loss: 37.7591 | Val Loss: 17978890.0000
Epoch 11/100 | Train Loss: 36.2601 | Val Loss: 18045612.0000
Epoch 12/100 | Train Loss: 35.0750 | Val Loss: 17974234.0000
Epoch 13/100 | Train Loss: 34.0395 | Val Loss: 17994422.0000
Epoch 14/100 | Train Loss: 32.7986 | Val Loss: 17952390.0000
Epoch 15/100 | Train Loss: 31.8368 | Val Loss: 17960896.0000
Epoch 16/100 | Train Loss: 31.0003 | Val Loss: 17961918.0000
Epoch 17/100 | Train Loss: 30.195

In [31]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [32]:
model.eval()

with torch.no_grad():
    train_preds=model(X_train_tensor)
    test_preds=model(X_test_tensor)
    train_mse_loss=crieterion(train_preds,y_train_tensor)
    test_mse_loss=crieterion(test_preds,y_test_tensor)

print(train_mse_loss.item())
print(test_mse_loss.item())

21.131654739379883
16507541.0
